## Part 6 of 6: Evaluating the Models

Loads fitted models and test scores from notebooks/05_model_building.ipynb. Last notebook in the sequence.

See `notebooks/README.md` for the full run order.

In [ ]:
# Import Python libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_recall_curve, precision_score, recall_score)
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Libraries have been imported!")

In [ ]:
# Load state saved by the previous notebook
import joblib
_state = joblib.load("_state/05_state.joblib")
globals().update(_state)
print(f"Loaded {len(_state)} objects: {sorted(_state)}")


In [ ]:
# Showing confusion matrices for all five models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

results_df = pd.DataFrame(results).set_index("model")
KEEPER_MODELS = [m for m in ["LogReg (balanced)", "Altman Z", "XGBoost (primary)",
                  "Decision Tree (depth=3)", "Random Forest"] if m in results_df.index]

fig, axes = plt.subplots(1, len(KEEPER_MODELS), figsize=(4*len(KEEPER_MODELS), 4))
for ax, name in zip(axes, KEEPER_MODELS):
    thr = results_df.loc[name, "threshold"]
    pred = (test_scores[name] >= thr).astype(int)
    cm = confusion_matrix(y_te, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Alive", "Failed"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues", values_format="d")
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluating the models using metrics table and Precision-Recall (PR-AUC) curves
res = pd.DataFrame(results).set_index("model")
print(f"Test set (2015-2018): {len(y_te):,} rows, {y_te.sum()} positives "
      f"({y_te.mean():.2%}). Baseline PR-AUC for a random model = {y_te.mean():.4f}")
print(res.round(4).to_string())

# Plot the curves
plt.figure(figsize=(8, 6))
for name, sc in test_scores.items():
    p, r, _ = precision_recall_curve(y_te, sc)
    plt.plot(r, p, drawstyle="steps-post",
             label=f"{name} (PR-AUC={average_precision_score(y_te, sc):.3f})")
plt.axhline(y_te.mean(), color="gray", ls=":", label=f"chance ({y_te.mean():.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("Precision-recall curves, test set 2015-2018")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# Plot ROC curves and AUC on the test set
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for name, sc in test_scores.items():
    fpr, tpr, _ = roc_curve(y_te, sc)
    plt.plot(fpr, tpr, label=f"{name} (ROC-AUC={roc_auc_score(y_te, sc):.3f})")
plt.plot([0, 1], [0, 1], color="gray", ls=":", label="chance (0.500)")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves, test set 2015-2018")
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
# Explainability by using SHAP on the test set
import shap
explainer = shap.TreeExplainer(xgb)
sv = explainer(X_te)

# 1. Global importance bar chart
shap.plots.bar(sv, max_display=15, show=False)
plt.title("Global feature importance (mean |SHAP|, test set)")
plt.tight_layout(); plt.show()

# 2. Beeswarm
shap.plots.beeswarm(sv, max_display=15, show=False)
plt.title("SHAP beeswarm, test set")
plt.tight_layout(); plt.show()

# 3. Waterfall for a correctly predicted bankruptcy (core Streamlit deliverable)
correct_pos = np.where((y_te.values == 1) &
                       (test_scores["XGBoost (primary)"] >= xgb_thr))[0]
if len(correct_pos):
    i = correct_pos[np.argmax(test_scores["XGBoost (primary)"][correct_pos])]
    firm, yr = te.iloc[i][["company_name", "year"]]
    print(f"SHAP waterfall | {firm}, data year {int(yr)} (filed {int(yr) + 1}), "
          f"predicted prob {test_scores['XGBoost (primary)'][i]:.3f}")
    shap.plots.waterfall(sv[i], max_display=12, show=False)
    plt.tight_layout(); plt.show()
else:
    print("No correctly predicted bankruptcies at the frozen threshold; "
          "inspect the highest-scored true positives manually.")

In [ ]:
# Feature importance for Altman Z-Score

altman_coefs = {"altman_x1": 1.2, "altman_x2": 1.4, "altman_x3": 3.3,
                "altman_x4": 0.6, "altman_x5": 1.0}

train_means = X_trva[list(altman_coefs.keys())].mean()

altman_contribs = pd.DataFrame({
    name: altman_coefs[name] * (te[name] - train_means[name])
    for name in altman_coefs
})
altman_importance = altman_contribs.abs().mean().sort_values(ascending=False)

# Plot feature importance
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(altman_importance.index[::-1], altman_importance.values[::-1], color="#4C72B0")
ax.set_xlabel("mean(|contribution to Z|), test set")
ax.set_title("Altman Z-Score Feature importance")
for i, v in enumerate(altman_importance.values[::-1]):
    ax.text(v, i, f" +{v:.2f}", va="center")
plt.tight_layout(); plt.show()

# Show fixed weights for reference
print("Altman's fixed weights, for reference:")
print(pd.Series(altman_coefs).sort_values(ascending=False).to_string())

In [ ]:
# Comparing feature importance in 1968 vs today
today_importance_all = pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES)
today_five = today_importance_all.loc[list(altman_coefs.keys())]

LABELS = {"altman_x1": "X1: Working capital / TA",
          "altman_x2": "X2: Retained earnings / TA",
          "altman_x3": "X3: EBIT / TA",
          "altman_x4": "X4: Market equity / liabilities",
          "altman_x5": "X5: Sales / TA"}

# Normalize both sides to % share of the 5 Altman variables, since raw
# SHAP values and Altman's raw coefficients live on different scales
today_pct = today_five / today_five.sum() * 100
altman_pct = altman_importance / altman_importance.sum() * 100

order = today_pct.sort_values(ascending=True).index
labels = [LABELS[n] for n in order]
y = np.arange(len(order))
h = 0.35

# Plot the results
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(y - h/2, altman_pct[order], height=h, color="#B0B0B0", label="1968 (Altman weights)")
ax.barh(y + h/2, today_pct[order], height=h, color="#4C72B0", label="Today (XGBoost mean SHAP)")

for i, n in enumerate(order):
    ax.text(altman_pct[n] + 1, i - h/2, f"{altman_pct[n]:.0f}%", va="center", fontsize=9, color="#666666")
    ax.text(today_pct[n] + 1, i + h/2, f"{today_pct[n]:.0f}%", va="center", fontsize=9, color="#4C72B0")

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Share of importance among the 5 Altman variables (%)")
ax.set_title("Which Altman input matters most: 1968 vs. today", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", frameon=False, fontsize=9)
ax.set_xlim(0, max(altman_pct.max(), today_pct.max()) + 10)
plt.tight_layout(); plt.show()

In [ ]:
# Probability calibration model to turn our predictions into percentages
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss

tuning_model = best["model"]
raw_probs_te = tuning_model.predict_proba(X_te)[:, 1]

prob_true, prob_pred = calibration_curve(y_te, raw_probs_te, n_bins=10, strategy="quantile")
print(f"Raw Brier score: {brier_score_loss(y_te, raw_probs_te):.4f}")

# Fit a Platt scaling (sigmoid) map on validation only, using the
# tuning-stage model specifically, since it never trained on validation.
calibrated = CalibratedClassifierCV(FrozenEstimator(tuning_model), method="sigmoid")
calibrated.fit(X_va, y_va)
calibrated_probs_te = calibrated.predict_proba(X_te)[:, 1]

prob_true_cal, prob_pred_cal = calibration_curve(y_te, calibrated_probs_te, n_bins=10, strategy="quantile")
print(f"Brier score (calibrated): {brier_score_loss(y_te, calibrated_probs_te):.4f}")

# Plot the graph
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(prob_pred, prob_true, marker="o", label="Raw XGBoost")
ax.plot(prob_pred_cal, prob_true_cal, marker="o", label="Calibrated (Platt scaling)")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed fraction of actual failures")
ax.set_title("Calibration curve, before vs after")
ax.legend()
plt.tight_layout(); plt.show()